# Avaliação de veredito com LLM local

Use uma GPU L4. Envie `data/eval/local_llm_verdict_dev_v1.jsonl` para uma comparação reprodutível ou `evals/golden_claims.jsonl` para explorar o conjunto canônico. Não use o held-out v1 para ajustar modelo ou prompt.


In [ ]:
%pip -q install 'transformers>=4.45' accelerate bitsandbytes


In [ ]:
from google.colab import files
uploaded = files.upload()
input_path = next(iter(uploaded))


In [ ]:
import json, re, time, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
# Para o controle 7B, troque os três valores pelo modelo/revisão/nome de saída correspondentes.
MODEL_ID = 'Qwen/Qwen2.5-14B-Instruct'
MODEL_REVISION = 'cf98f3b'
OUTPUT_NAME = 'local_llm_qwen25_14b_dev_v1.json'
assert torch.cuda.is_available(), 'Selecione uma GPU L4 no Colab.'
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=MODEL_REVISION, quantization_config=quant, device_map='auto')
model.eval()
items = [json.loads(line) for line in open(input_path, encoding='utf-8') if line.strip()]
print({'model': MODEL_ID, 'revision': MODEL_REVISION, 'device': torch.cuda.get_device_name(0), 'n': len(items)})


In [ ]:
SYSTEM = 'Você é um classificador de checagem de fatos em português. Use somente o laudo. Responda somente JSON: {\"conclusion\": \"sustentada|contradita|inconclusiva\"}. Sustentada exige suporte direto; contradita exige contradição explícita da alegação inteira. Ausência de prova, evidência parcial, entidade, número ou período diferente e dados insuficientes são inconclusivos, nunca contradita.'
def evidence_fields(item):
    evidence = item.get('evidence', {})
    return item.get('evidence_title', evidence.get('titulo', '')), item.get('evidence_text', evidence.get('texto', ''))

def predict(item):
    title, text = evidence_fields(item)
    user = f"ALEGAÇÃO:\n{item['claim']}\n\nLAUDO — {title}:\n{text}"
    messages = [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': user}]
    encoded = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors='pt', return_dict=True).to(model.device)
    input_length = encoded['input_ids'].shape[-1]
    with torch.inference_mode():
        generated = model.generate(**encoded, max_new_tokens=32, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    raw = tokenizer.decode(generated[0][input_length:], skip_special_tokens=True).strip()
    match = re.search(r'\"conclusion\"\s*:\s*\"(sustentada|contradita|inconclusiva)\"', raw)
    return (match.group(1) if match else 'invalida'), raw

started = time.perf_counter(); rows = []
for item in items:
    pred, raw = predict(item)
    rows.append({'id': item['id'], 'gold': item['gold_conclusion'], 'pred': pred, 'tags': item['tags'], 'raw': raw})
elapsed = time.perf_counter() - started
print(f'{len(rows)} casos processados em {elapsed:.1f}s')


In [ ]:
labels = ['sustentada', 'contradita', 'inconclusiva']
accuracy = sum(row['gold'] == row['pred'] for row in rows) / len(rows)
scores = []
for label in labels:
    tp = sum(row['gold'] == label and row['pred'] == label for row in rows)
    fp = sum(row['gold'] != label and row['pred'] == label for row in rows)
    fn = sum(row['gold'] == label and row['pred'] != label for row in rows)
    precision = tp / (tp + fp) if tp + fp else 0
    recall = tp / (tp + fn) if tp + fn else 0
    scores.append(2 * precision * recall / (precision + recall) if precision + recall else 0)
report = {'model': MODEL_ID, 'model_revision': MODEL_REVISION, 'device': torch.cuda.get_device_name(0), 'n': len(rows), 'accuracy': accuracy, 'macro_f1': sum(scores) / len(scores), 'invalid_outputs': sum(row['pred'] == 'invalida' for row in rows), 'elapsed_seconds': elapsed, 'cases_per_second': len(rows) / elapsed, 'cases': rows, 'scope': 'Development-only local LLM screening; not a production decision.'}
open(OUTPUT_NAME, 'w', encoding='utf-8').write(json.dumps(report, ensure_ascii=False, indent=2))
print(json.dumps({key: value for key, value in report.items() if key != 'cases'}, ensure_ascii=False, indent=2))
files.download(OUTPUT_NAME)
